# 03 · 城市天际线-动手做(只调高度)
**锁死 footprint 与角色标签,只放开高度这一个自由度**,看权力怎么改写天际线。
和新加坡 OSM 版同一手感——但跑**上海真实数据、实测高度**(陆家嘴最高 338m)。不用写代码,逐格执行即可。

> 你可以改三处(都用记事本就能开):
> - `config.py` 的 `SLUG` —— 换站点(lujiazui / caoyang / yuyuan)。改完重跑下面「准备」格。
> - `shanghai_lookup.yaml` —— 谁算谁的。改完重跑 **Step 2**。
> - `power_scenarios.yaml` —— 每个角色的高度政策(mult / cap_m / _mode)。改完重跑 **Step 3、4**。

## 怎么执行
- 点一格,按 **Shift+Enter** 执行;或选单 Run → Run All Cells 从头跑。
- 结果与图会显示在该格下面。代码都在 `engine/` 文件夹,平时不用打开。

In [ ]:
# 让 notebook 找到 engine 里的代码(这格不用改,直接执行)
import sys, os
sys.path.insert(0, os.path.abspath("engine"))
import importlib, config; importlib.reload(config)
import common as C, plots
print("就绪 ·  当前站点 SLUG =", config.SLUG, "(", config.site_name(), ")")

In [ ]:
df = C.current_buildings(config.SLUG)
df = C.assign_all(df)           # 贴上 stakeholder(改 shanghai_lookup.yaml 可变)
print("%s · %d 栋 · 最高 %.0fm" % (config.site_name(), len(df), df.height_m.max()))

## Step 0(可选):先看真实的地方
卫星底图 + figure-ground(把我们的离散读法叠在真实卫星上)。需要 `contextily` 和网络,没有就跳过。

In [ ]:
try:
    plots.satellite_figureground(df)     # 需联网;失败自动跳过
except Exception as e:
    print("跳过卫星(没网或缺 contextily):", e)

## Step 1:读建筑(实测高度)
这步**还没上色**,全灰,先看清原料:这批楼长什么形状、有多高。上海版高度**全是 AI 实测**,
不是楼层估算——所以才有真正的超高层 token(右图 >100m 的红色)。

In [ ]:
plots.data_overview(df)

## Step 2:贴角色标签
按 `shanghai_lookup.yaml` 的级联查表,给每栋一个角色(详见「02 映射」那本)。只加标签,不动几何与高度。

In [ ]:
print("各角色栋数:", df.stakeholder.value_counts().to_dict())
plots.power_map(df)

## Step 3:权力情景(高度政策面板)
每个情景给各角色一个高度权重 `mult`、可选上限 `cap_m`,与模式 `_mode`:
- **conserve**(默认):总建筑量守恒,只在角色之间**重分配**。
- **grow**:以现有高度为地板,**只增不减**(都更新增),总量上升。

In [ ]:
scen = C.load_scenarios()
print("可用情景:", list(scen.keys()))
plots.policy_heatmap(scen)      # 红=拔高 / 蓝=压低 / 白=照旧

## Step 4:套情景,只调高度 —— PAYOFF
同一批楼、同样 footprint 与标签,分别套几个情景**只改高度**。因为 footprint 与标签锁定,
天际线差异**完全来自权力配置**。conserve 总量守恒、只此消彼长;想看大变化用 grow 情景(如 `developer_renewal`)。

In [ ]:
SCEN_NAMES = ["current", "developer_led", "community_led", "state_eco"]   # ✏️ 可改要比的情景
heights = {n: (C.scenario_heights(df, scen[n]) if n != "current" else df["height_m"]) for n in SCEN_NAMES}
plots.skyline_panels(df, heights)    # 4 连幅,同色阶可横比
plots.metrics(df, heights)           # 总GFA / 平均高 / 最高 / 各角色平均高

## Step 5:出 3D 量体 + OBJ(给 Rhino/GH)
把 footprint 按高度挤成 3D。屏幕预览取占地最大的若干栋(画得快);OBJ 用全部楼、写到 `out/<slug>/`。

In [ ]:
from pathlib import Path
outdir = C.OUT / config.SLUG; outdir.mkdir(parents=True, exist_ok=True)
sub = df.sort_values("area_m2", ascending=False).head(120).copy()
plots.city_3d(sub)                                  # 屏幕预览(matplotlib)
obj, nv, nf = C.extrude_obj(df, height_col="height_m")
(outdir / "city_current.obj").write_text(obj, encoding="utf-8")
print("OBJ 写到 %s/city_current.obj  · verts %d faces %d" % (config.SLUG, nv, nf))

## 动手练习
改一处、重跑对应步骤,看天际线变化:
- 换站点 → `config.py` 的 `SLUG`(陆家嘴↔曹杨↔豫园)→ 重跑「准备」格起。
- 改谁算谁的 → `shanghai_lookup.yaml` → 重跑 Step 2。
- 改高度政策 / 加情景 → `power_scenarios.yaml` → 重跑 Step 3、4。

> 想让权力不止改高度、而是改**形态语法**(拆板成塔、细粒自建、向权力重心收拢)?
> 进下一本:[`04 进阶-权力算子`]。